# 🐐 Animal Disease Models: Mobile Anemia Detection Pipeline
### Fast 1-Click GPU Training & TFLite Mobile Export for Sheep & Goat Anemia Diagnosis

This Google Colab notebook trains the two mobile AI models:
1. **Model 1: The 'Finder' (YOLOv8 Nano)** — Detects and isolates inner eyelids.
2. **Model 2: The 'Judge' (MobileNetV3 Small)** — Classifies anemia severity (Green / Yellow / Red).
3. **Exports directly to `.tflite`** for offline deployment in your mobile app.

---
### Step 0: Ensure GPU is Enabled
Go to **Runtime > Change runtime type** and select **T4 GPU**.

In [ ]:
# Check GPU availability
!nvidia-smi

### Step 1: Clone Repository & Install Dependencies
Clones the project files and enters the directory inside Google Colab.

In [ ]:
import os
# 1. Clone repository into Google Colab if not already inside
if os.path.basename(os.getcwd()) != 'Animal-Disease-Models':
    if not os.path.exists('Animal-Disease-Models'):
        !git clone https://github.com/AsemaniJoshua/Animal-Disease-Models.git
    %cd Animal-Disease-Models

# 2. Pull latest code
!git pull origin main

# 3. Install required libraries
!pip install -q ultralytics torchvision torch pandas openpyxl onnx onnxruntime onnxslim onnx2tf onnxscript
print(f"Current working directory: {os.getcwd()}")
print("Repository cloned and dependencies installed successfully!")

### Step 2: Prepare Datasets
Organize YOLO eye detection and CP-AnemiC conjunctiva images into training and validation splits.

In [ ]:
# Run data preparation scripts
!python data/prepare_yolo_data.py
!python data/prepare_judge_data.py

### Step 3: Train Model 1 (The Finder / YOLOv8 Nano Eye Detector)

In [ ]:
from ultralytics import YOLO

model1 = YOLO("yolov8n.pt")
results1 = model1.train(
    data="configs/dataset_yolo.yaml",
    epochs=30,
    imgsz=640,
    batch=32,
    device=0,
    project="runs/finder",
    name="train",
    exist_ok=True
)
print("Model 1 (Finder) training complete!")

### Step 4: Train Model 2 (The Judge / MobileNetV3 Anemia Classifier)

In [ ]:
!python training/train_judge.py --epochs 30 --batch 32 --device cuda

### Step 5: Export Both Models to `.tflite` for Mobile App

In [ ]:
import shutil
import glob
from pathlib import Path
from ultralytics import YOLO

# 1. Export Model 1 to TFLite
print("Exporting Model 1 (Finder) to TFLite...")
finder_model = YOLO("runs/finder/train/weights/best.pt")
finder_tflite = finder_model.export(format="tflite", imgsz=640)
dest_finder_tflite = Path("exported_models/finder_yolo.tflite")
shutil.copy2(finder_tflite, dest_finder_tflite)
print(f"✓ Finder TFLite: {dest_finder_tflite} ({dest_finder_tflite.stat().st_size / (1024*1024):.2f} MB)")

# 2. Export Model 2 to ONNX and convert to TFLite
print("\nExporting Model 2 (Judge)...")
!python export/export_judge.py

# Convert Model 2 ONNX to TFLite
!onnx2tf -in exported_models/judge_mobilenet.onnx -o exported_models/judge_tflite_out -cotof
judge_tflites = glob.glob("exported_models/judge_tflite_out/*.tflite")
if judge_tflites:
    dest_judge_tflite = Path("exported_models/judge_mobilenet.tflite")
    shutil.copy2(judge_tflites[0], dest_judge_tflite)
    print(f"✓ Judge TFLite: {dest_judge_tflite} ({dest_judge_tflite.stat().st_size / (1024*1024):.2f} MB)")

print("\nExported mobile files in exported_models/:")
!ls -lh exported_models

### Step 6: Test End-to-End Simulation

In [ ]:
!python pipeline/end_to_end_demo.py --lang hausa

### Step 7: Download Models for Mobile App
Zip and download your `.tflite` models directly to copy into your mobile repo's `assets/models/` folder.

In [ ]:
from google.colab import files
!zip -r mobile_models.zip exported_models/
files.download("mobile_models.zip")
print("Download started! Extract mobile_models.zip and place into your mobile app repo assets.")